In [1]:
import pandas as pd

In [2]:
# Import the CSV, convert both date columns to datetime, and create a column called Resolution_Hours that calculates how many hours it took to resolve each ticket (as a decimal, rounded to 1 decimal place).
df = pd.read_csv('support_tickets.csv')
df['Created_Date'] = pd.to_datetime(df['Created_Date'])
df['Resolved_Date'] = pd.to_datetime(df['Resolved_Date'])
df['Resolution_Hours'] = (((df['Resolved_Date'] - df['Created_Date']).dt.total_seconds() / 3600).round(1))
df

,Ticket_ID,Customer_ID,Agent_ID,Created_Date,Resolved_Date,Category,Priority,Satisfaction_Rating,Resolution_Hours
0,T001,C101,A01,2024-01-03 09:00:00,2024-01-03 14:30:00,Billing,High,4,5.5
1,T002,C102,A02,2024-01-03 10:15:00,2024-01-04 11:00:00,Technical,Critical,3,24.8
2,T003,C101,A01,2024-01-05 08:00:00,2024-01-05 09:45:00,Billing,Low,5,1.8
3,T004,C103,A03,2024-01-06 13:00:00,2024-01-08 10:00:00,Technical,Critical,2,45.0
4,T005,C104,A01,2024-01-07 11:30:00,2024-01-07 16:00:00,General,Medium,4,4.5
5,T006,C102,A02,2024-01-08 09:00:00,2024-01-08 17:30:00,Billing,High,3,8.5
6,T007,C105,A03,2024-01-09 14:00:00,2024-01-11 09:00:00,Technical,Critical,1,43.0
7,T008,C101,A01,2024-01-10 10:00:00,2024-01-10 11:30:00,General,Low,5,1.5
8,T009,C106,A02,2024-01-10 15:00:00,2024-01-12 14:00:00,Technical,High,2,47.0
9,T010,C103,A03,2024-01-11 08:30:00,2024-01-11 16:00:00,Billing,Medium,4,7.5


In [3]:
# Create a pivot table that shows the average Resolution_Hours for each Category (rows) by each Priority level (columns). Fill any missing combinations with 0.
pd.pivot_table(df, values = 'Resolution_Hours', index = 'Category', columns = 'Priority', aggfunc = 'mean', fill_value=0).round(1)

Priority,Critical,High,Low,Medium
Category,,,,
Billing,0.0,6.5,1.8,5.7
General,0.0,0.0,1.0,4.2
Technical,45.6,49.0,0.0,0.0


In [4]:
# For each Agent_ID, calculate their total tickets handled, average Satisfaction_Rating, and average Resolution_Hours. Then rank the agents by Satisfaction_Rating descending. Add the rank as a column.
result = df.groupby('Agent_ID').agg(Number_of_Tickets = ('Ticket_ID', 'size'), Avg_Rating = ('Satisfaction_Rating', 'mean'), Avg_Resolution_Time = ('Resolution_Hours', 'mean')).reset_index()
result['Rank'] = result['Avg_Rating'].rank(ascending=False).astype(int)
result = result.sort_values('Rank')
result

,Agent_ID,Number_of_Tickets,Avg_Rating,Avg_Resolution_Time,Rank
0,A01,8,4.625000,2.500000,1
2,A03,6,2.833333,18.416667,2
1,A02,6,2.166667,41.050000,3


In [5]:
# Identify repeat customers — customers who submitted more than one ticket. For those repeat customers only, calculate their average Satisfaction_Rating and total number of tickets. Display sorted by total tickets descending.
repeatcust = df.groupby('Customer_ID').agg(Avg_Satisfaction_Rating = ('Satisfaction_Rating', 'mean'), Number_of_Tickets = ('Customer_ID','size')).round(1)
repeatcust = repeatcust[repeatcust['Number_of_Tickets'] > 1]
repeatcust = repeatcust.sort_values('Number_of_Tickets', ascending = False)
repeatcust

,Avg_Satisfaction_Rating,Number_of_Tickets
Customer_ID,,
C101,4.8,4
C102,3.7,3
C103,3.0,3
C104,3.0,2


In [6]:
# Create a column called SLA_Met that flags whether the ticket was resolved within the SLA target. The SLA targets by Priority are:
# Critical — 48 hours, High — 24 hours, Medium — 12 hours, Low — 8 hours. Label it 'Yes' if resolved within the target, 'No' if not. Then calculate the SLA compliance rate by Priority as a percentage rounded to 1 decimal place.
sla_target = {'Critical' : 48, 'High' : 24, 'Medium' : 12, 'Low' : 8}
df['SLA_Target'] = df['Priority'].map(sla_target)
df['SLA_Met'] = (df['Resolution_Hours'] <= df['SLA_Target'])
compliance = (df.groupby('Priority')['SLA_Met'].mean()* 100).round(1).reset_index(name = 'SLA_Compliance_Rate')
df['SLA_Met'] = (df['SLA_Met']).apply(lambda x: 'Yes' if x else 'No')
compliance

,Priority,SLA_Compliance_Rate
0,Critical,80.0
1,High,60.0
2,Low,100.0
3,Medium,100.0
